# 第 8 章 · Bellman 期望方程：今天的价值 = 眼前的奖励 + γ × 明天的价值

本章回答：不跑几千次，能不能直接算出 $v_\pi$？各个状态的价值之间有什么关系？书上那张 5×5 网格的数字是怎么来的？

配套讲义：仓库里 `cs285-deep-rl/notes/lessons/08-bellman.md`（讲解、动画、练习都在讲义里）。

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU。

In [ ]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)  # 打印保留 3 位小数
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 代码 | 跑出来 |
|---|---|---|
| 机器人两个方程 | `solve_v(P, pi, 0.9)` | 15.828、13.869 |
| 代回去验证 | `bellman_rhs(P, pi, v, 0.9, s)` | 右边也是 15.828、13.869 |
| 网格 25 个方程 | `solve_v(G, uniform_policy(G), 0.9)` | 图 3.2 右（A 8.8，B 5.3） |

## 1. 先把要用的东西定义好

这几格只是定义（环境、函数），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**常量 `ROBOT_PI`**（来自 `rl_envs.py`）

In [ ]:
ROBOT_PI = {0: {0: 0.9, 1: 0.1}, 1: {0: 0.2, 1: 0.3, 2: 0.5}}

**常量 `ROBOT_STATES`**（来自 `rl_envs.py`）

In [ ]:
ROBOT_STATES = ["高电", "低电"]

**函数 `q_from_v`**（来自 `ch08_bellman.py`）：q_π(s, a) = Σ p(s', r|s, a) [r + γ v_π(s')]。

In [ ]:
def q_from_v(P, v, gamma, s, a):
    """q_π(s, a) = Σ p(s', r|s, a) [r + γ v_π(s')]。"""
    return sum(p * (r + (0 if done else gamma * v[ns])) for p, ns, r, done in P[s][a])

**函数 `bellman_rhs`**（来自 `ch08_bellman.py`）：下面要用到的小工具。

In [ ]:
def bellman_rhs(P, pi, v, gamma, s):
    return sum(pa * q_from_v(P, v, gamma, s, a) for a, pa in pi[s].items())

**常量 `MOVES`**（来自 `rl_envs.py`）

In [ ]:
MOVES = [(-1, 0), (1, 0), (0, 1), (0, -1)]

**函数 `gridworld_5x5`**（来自 `rl_envs.py`）：S&B Example 3.5（图 3.2）：5×5，A=(0,1) 任何动作 +10 跳到 A'=(4,1)，B=(0,3) +5 跳到 B'=(2,3)，

In [ ]:
def gridworld_5x5():
    """S&B Example 3.5（图 3.2）：5×5，A=(0,1) 任何动作 +10 跳到 A'=(4,1)，B=(0,3) +5 跳到 B'=(2,3)，
    撞墙原地不动 −1，其余 0。连续任务，不结束。状态编号 = 行 × 5 + 列。"""
    n = 5
    P = {}
    for r in range(n):
        for c in range(n):
            s = r * n + c
            P[s] = {}
            for a, (dr, dc) in enumerate(MOVES):
                if (r, c) == (0, 1):
                    P[s][a] = [(1.0, 4 * n + 1, 10.0, False)]
                elif (r, c) == (0, 3):
                    P[s][a] = [(1.0, 2 * n + 3, 5.0, False)]
                else:
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < n and 0 <= nc < n:
                        P[s][a] = [(1.0, nr * n + nc, 0.0, False)]
                    else:
                        P[s][a] = [(1.0, s, -1.0, False)]
    return P

**常量 `ALPHA`, `BETA`**（来自 `rl_envs.py`）

In [ ]:
ALPHA, BETA = 0.8, 0.7

**常量 `R_SEARCH`, `R_WAIT`, `R_RESCUE`**（来自 `rl_envs.py`）

In [ ]:
R_SEARCH, R_WAIT, R_RESCUE = 2.0, 1.0, -3.0

**函数 `robot_mdp`**（来自 `rl_envs.py`）：返回 P[s][a]；状态 0 = 高电、1 = 低电；动作 0 = 搜索、1 = 等待、2 = 充电（只有低电能充电）。

In [ ]:
def robot_mdp():
    """返回 P[s][a]；状态 0 = 高电、1 = 低电；动作 0 = 搜索、1 = 等待、2 = 充电（只有低电能充电）。"""
    hi, lo = 0, 1
    P = {
        hi: {
            0: [(ALPHA, hi, R_SEARCH, False), (1 - ALPHA, lo, R_SEARCH, False)],
            1: [(1.0, hi, R_WAIT, False)],
        },
        lo: {
            0: [(BETA, lo, R_SEARCH, False), (1 - BETA, hi, R_RESCUE, False)],
            1: [(1.0, lo, R_WAIT, False)],
            2: [(1.0, hi, 0.0, False)],
        },
    }
    return P

**函数 `policy_matrices`**（来自 `ch08_bellman.py`）：把 (P, π) 压成 r_π[s] = 期望即时奖励、P_π[s, s'] = 一步转移概率（终点之后不再有价值）。

In [ ]:
def policy_matrices(P, pi):
    """把 (P, π) 压成 r_π[s] = 期望即时奖励、P_π[s, s'] = 一步转移概率（终点之后不再有价值）。"""
    n = len(P)
    r_pi, P_pi = np.zeros(n), np.zeros((n, n))
    for s in P:
        for a, pa in pi[s].items():
            for p, ns, r, done in P[s][a]:
                r_pi[s] += pa * p * r
                if not done:
                    P_pi[s, ns] += pa * p
    return r_pi, P_pi

**函数 `solve_v`**（来自 `ch08_bellman.py`）：v = r_π + γ P_π v  ⇒  (I − γ P_π) v = r_π。

In [ ]:
def solve_v(P, pi, gamma):
    """v = r_π + γ P_π v  ⇒  (I − γ P_π) v = r_π。"""
    r_pi, P_pi = policy_matrices(P, pi)
    return np.linalg.solve(np.eye(len(P)) - gamma * P_pi, r_pi)

**函数 `uniform_policy`**（来自 `rl_envs.py`）：下面要用到的小工具。

In [ ]:
def uniform_policy(P):
    return {s: {a: 1.0 / len(P[s]) for a in P[s]} for s in P}

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
P, pi = robot_mdp(), ROBOT_PI

In [ ]:
v = solve_v(P, pi, 0.9)

In [ ]:
for s in P:
        print(f"v_π({ROBOT_STATES[s]}) = {v[s]:.3f}；Bellman 右边 = {bellman_rhs(P, pi, v, 0.9, s):.3f}")

In [ ]:
G = gridworld_5x5()

In [ ]:
vg = solve_v(G, uniform_policy(G), 0.9)

In [ ]:
print("5×5 网格、随机策略（图 3.2 右）：")

In [ ]:
print(np.round(vg.reshape(5, 5), 1))

## 3. 代码里才看得到的

`np.linalg.solve` 比先求逆再相乘更快也更准，解方程组时一律用它。第 10 章会看到另一种算法：不解方程，而是从 0 开始反复代入式 3.14，数会一轮轮逼近同一个答案。

## 4. 自己改一改

讲义里建议的几个小实验（数字是实跑出来的，改完对一对）：

1. `solve_v(P, pi, 0.5)` → [3.49, 1.766]，和第 7 章跑很多局估的 3.483 对得上。
2. 把低电的策略改成"必充电"（`{0: 0.0, 1: 0.0, 2: 1.0}`），`solve_v(P, pi2, 0.9)` → [16.351, 14.716]，两个状态都比原来的 15.828、13.869 高：换策略，价值就跟着变。

讲义末尾的练习题也可以在下面的空格子里用代码验算：先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 今天的价值 = 眼前奖励 + γ × 明天的价值，再取平均；
> 先按 p 平均成 q，再按 π 平均成 v；
> 一个状态一个方程，知道 p 就能解；
> A 格八点八：拿了十块被扔到墙边。